# 05. 소리 샘플 — 20 클래스 + 노이즈

20개 foreground 클래스와 TAU 도시 소음을 각각 1개씩 재생/시각화하는 노트북.

**전제 조건**: `data/BinauralCuratedDataset_mini/` 가 존재해야 합니다 (또는 `BinauralCuratedDataset/`).

준비:
```bash
bash scripts/eval/eval_mini.sh ./data ./eval_results/mini
```

노트북 셀 출력은 의도적으로 비워둠 (Run All 로 실행).

## Setup

In [ ]:
import os
from pathlib import Path

import numpy as np
import soundfile as sf
import librosa
import librosa.display
import matplotlib.pyplot as plt
from IPython.display import Audio, display

# 데이터 루트: mini 우선, 없으면 full dataset
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'configs').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

DATA_ROOT_CANDIDATES = [
    REPO_ROOT / 'data' / 'BinauralCuratedDataset_mini',
    REPO_ROOT / 'data' / 'BinauralCuratedDataset',
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if p.exists()), None)
assert DATA_ROOT is not None, f'Dataset not found in: {DATA_ROOT_CANDIDATES}'
print(f'REPO_ROOT: {REPO_ROOT}')
print(f'DATA_ROOT: {DATA_ROOT}')

FG_DIR = DATA_ROOT / 'scaper_fmt' / 'test'
NOISE_DIR = DATA_ROOT / 'noise_scaper_fmt' / 'test'
assert FG_DIR.exists(), FG_DIR
assert NOISE_DIR.exists(), NOISE_DIR

In [ ]:
# 20 foreground 클래스 (alphabetical)
FG_CLASSES = sorted([d.name for d in FG_DIR.iterdir() if d.is_dir()])
print(f'FG classes ({len(FG_CLASSES)}):')
for c in FG_CLASSES:
    print(f'  - {c}')
assert len(FG_CLASSES) == 20, f'Expected 20 classes, got {len(FG_CLASSES)}'

In [ ]:
# 헬퍼: 클래스에서 첫 번째 파일 1개 선택 (deterministic)
def first_audio_in(class_dir: Path) -> Path:
    files = sorted(p for p in class_dir.iterdir() if p.suffix.lower() in ('.wav', '.flac'))
    assert files, f'No audio files in {class_dir}'
    # 심볼릭 링크인 경우 resolve
    return files[0].resolve() if files[0].is_symlink() else files[0]

# 헬퍼: load + 표시
def show_sample(label: str, audio_path: Path, target_sr: int = 16000):
    y, sr = sf.read(str(audio_path))
    if y.ndim > 1:
        y = y.mean(axis=1)  # mono downmix
    if sr != target_sr:
        y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=target_sr)
        sr = target_sr

    duration = len(y) / sr
    print(f'[{label}]  file={audio_path.name}  sr={sr}Hz  duration={duration:.2f}s  rms={np.sqrt((y**2).mean()):.4f}')

    # 스펙트로그램
    fig, axes = plt.subplots(1, 2, figsize=(11, 2.5))
    axes[0].plot(np.arange(len(y))/sr, y, linewidth=0.5)
    axes[0].set_title(f'{label} — waveform')
    axes[0].set_xlabel('time (s)')
    axes[0].set_xlim(0, max(duration, 1.0))

    S = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=1024, hop_length=256)), ref=np.max)
    librosa.display.specshow(S, sr=sr, hop_length=256, x_axis='time', y_axis='hz', ax=axes[1])
    axes[1].set_title(f'{label} — spectrogram (dB)')
    plt.tight_layout()
    plt.show()

    display(Audio(y, rate=sr))

## 20 Foreground 클래스 샘플

In [ ]:
show_sample('alarm_clock', first_audio_in(FG_DIR / 'alarm_clock'))

In [ ]:
show_sample('baby_cry', first_audio_in(FG_DIR / 'baby_cry'))

In [ ]:
show_sample('birds_chirping', first_audio_in(FG_DIR / 'birds_chirping'))

In [ ]:
show_sample('car_horn', first_audio_in(FG_DIR / 'car_horn'))

In [ ]:
show_sample('cat', first_audio_in(FG_DIR / 'cat'))

In [ ]:
show_sample('cock_a_doodle_doo', first_audio_in(FG_DIR / 'cock_a_doodle_doo'))

In [ ]:
show_sample('computer_typing', first_audio_in(FG_DIR / 'computer_typing'))

In [ ]:
show_sample('cricket', first_audio_in(FG_DIR / 'cricket'))

In [ ]:
show_sample('dog', first_audio_in(FG_DIR / 'dog'))

In [ ]:
show_sample('door_knock', first_audio_in(FG_DIR / 'door_knock'))

In [ ]:
show_sample('glass_breaking', first_audio_in(FG_DIR / 'glass_breaking'))

In [ ]:
show_sample('gunshot', first_audio_in(FG_DIR / 'gunshot'))

In [ ]:
show_sample('hammer', first_audio_in(FG_DIR / 'hammer'))

In [ ]:
show_sample('music', first_audio_in(FG_DIR / 'music'))

In [ ]:
show_sample('ocean', first_audio_in(FG_DIR / 'ocean'))

In [ ]:
show_sample('singing', first_audio_in(FG_DIR / 'singing'))

In [ ]:
show_sample('siren', first_audio_in(FG_DIR / 'siren'))

In [ ]:
show_sample('speech', first_audio_in(FG_DIR / 'speech'))

In [ ]:
show_sample('thunderstorm', first_audio_in(FG_DIR / 'thunderstorm'))

In [ ]:
show_sample('toilet_flush', first_audio_in(FG_DIR / 'toilet_flush'))

## TAU 도시 소음 (background noise)

`noise_scaper_fmt/test/` 의 첫 번째 폴더(scene)에서 1개 파일 선택.

In [ ]:
noise_scenes = sorted([d.name for d in NOISE_DIR.iterdir() if d.is_dir()])
print(f'TAU noise scenes ({len(noise_scenes)}):')
for s in noise_scenes[:10]:
    print(f'  - {s}')
if len(noise_scenes) > 10:
    print(f'  ... and {len(noise_scenes)-10} more')

# 첫 번째 scene 의 첫 파일
first_scene = NOISE_DIR / noise_scenes[0]
show_sample(f'tau_noise (scene={noise_scenes[0]})', first_audio_in(first_scene))

## (옵션) 학습/평가에서 실제로 사용되는 binaural mixture 합성 예시

`MisophoniaDataset.__getitem__` 을 직접 호출해 binaural mixture 한 샘플을 만들어 보기.

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT))

from src.datasets.MisophoniaDataset import MisophoniaDataset

ds = MisophoniaDataset(
    fg_sounds_dir=str(DATA_ROOT / 'scaper_fmt' / 'test'),
    bg_sounds_dir=str(DATA_ROOT / 'bg_scaper_fmt' / 'test'),
    noise_sounds_dir=str(DATA_ROOT / 'noise_scaper_fmt' / 'test'),
    hrtf_list=str(DATA_ROOT / 'hrtf' / 'CIPIC' / 'test_hrtf.txt'),
    split='test',
    sr=16000,
    duration=5,
    hrtf_type='CIPIC',
    samples_per_epoch=10,
    num_fg_sounds_min=1, num_fg_sounds_max=2,
    num_bg_sounds_min=1, num_bg_sounds_max=1,
    num_noise_sounds_min=1, num_noise_sounds_max=1,
    num_output_channels=5,
    snr_range_fg=[5, 15],
    snr_range_bg=[0, 10],
    use_torchaudio_resampling=False,  # 노트북에선 CPU resample 권장
)
print(f'len(ds) = {len(ds)}')
print(f'fg classes available: {ds.fg_sounds}')

In [ ]:
inputs, targets = ds[0]
print('inputs shapes:')
for k, v in inputs.items():
    print(f'  {k}: {tuple(v.shape) if hasattr(v, "shape") else v}')
print('targets:')
print(f'  target shape: {tuple(targets["target"].shape)}')
print(f'  fg_labels:    {targets["fg_labels"]}')
print(f'  bg_labels:    {targets["bg_labels"]}')
print(f'  noise_labels: {targets["noise_labels"]}')
print(f'  active label_vector indices: {(inputs["label_vector"] > 0).nonzero().squeeze().tolist()}')

In [ ]:
# Binaural mixture (2ch) 재생
mix = inputs['mixture'].numpy()  # (2, 80000)
sr = 16000
print(f'mixture shape: {mix.shape}, sr={sr}, duration={mix.shape[-1]/sr:.2f}s')
print('Binaural (stereo) playback:')
display(Audio(mix, rate=sr))

# 좌/우 채널 스펙트로그램
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for i, ax in enumerate(axes):
    S = librosa.amplitude_to_db(np.abs(librosa.stft(mix[i], n_fft=1024, hop_length=256)), ref=np.max)
    librosa.display.specshow(S, sr=sr, hop_length=256, x_axis='time', y_axis='hz', ax=ax)
    ax.set_title(f'mixture {["L", "R"][i]} channel')
plt.tight_layout(); plt.show()

In [ ]:
# Per-source ground truth (TSE 학습 타깃)
tgt = targets['target'].numpy()  # (5, 80000)
active_idx = [i for i, lbl in enumerate(targets['fg_labels']) if lbl != 'None']
print(f'active fg slots: {active_idx}, labels: {[targets["fg_labels"][i] for i in active_idx]}')

for slot in active_idx:
    print(f'\n--- target slot {slot}: {targets["fg_labels"][slot]} ---')
    display(Audio(tgt[slot], rate=sr))